# Module 11 — Exercises

These exercises follow the order of the concept guide: the likelihood and analytic MLEs, Fisher information,
numerical MLE, loss functions as negative log-likelihoods, conjugate posteriors, MAP as regularisation, the
posterior predictive, Bayesian A/B testing, and finally empirical-Bayes shrinkage. Several checks compare
your answer against a *hidden* value, so derive carefully rather than guessing.

**How these exercises work.** Each coding exercise has a *starter* cell (replace `# YOUR CODE HERE` with your implementation) followed by a *check* cell. Run the check cell to get an instant ✅/❌. Checks never stop the notebook, so you can run everything top-to-bottom at any time and see your score at the end. Some checks compare against a *hidden* answer (only a hash is stored), so you cannot peek at the number — think first, then verify. Conceptual questions hide their answer behind a *Show answer* toggle. Mini-projects are open-ended and come with a rubric. Difficulty is marked ★☆☆ (warm-up), ★★☆ (core), ★★★ (stretch).

### Setup and self-check helpers
Run the next two cells first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb
from scipy import stats, optimize, special
rng = np.random.default_rng(0)
plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
import hashlib as _hashlib
import numpy as _np

_results = {}

def _canon(value, decimals=3):
    """Canonical string form of an answer, so hashes are stable."""
    if isinstance(value, (bool, _np.bool_)):
        return "true" if value else "false"
    if isinstance(value, str):
        return " ".join(value.strip().lower().split())
    if isinstance(value, (int, _np.integer)):
        return str(int(value))
    if isinstance(value, (float, _np.floating)):
        s = f"{float(value):.{decimals}f}"
        return "0." + "0" * decimals if s in ("-0." + "0" * decimals, "-0") else s
    if isinstance(value, dict):
        return "{" + ",".join(f"{_canon(k, decimals)}:{_canon(v, decimals)}" for k, v in sorted(value.items(), key=lambda kv: str(kv[0]))) + "}"
    try:
        arr = _np.asarray(value)
        if arr.ndim == 0:
            return _canon(arr.item(), decimals)
        return "[" + ",".join(_canon(x, decimals) for x in arr.ravel().tolist()) + "]"
    except Exception:
        return str(value)

def sha_of(value, decimals=3):
    return _hashlib.sha256(_canon(value, decimals).encode()).hexdigest()[:16]

def check(name, condition, hint=""):
    """Record and print a pass/fail mark. Never raises, so the notebook keeps running."""
    try:
        ok = bool(condition)
    except Exception:
        ok = False
    _results[name] = ok
    print(("✅ " if ok else "❌ ") + name + ("" if ok or not hint else f"   — hint: {hint}"))
    return ok

def check_close(name, got, expected, tol=1e-6, hint=""):
    try:
        g = _np.asarray(got, dtype=float)
        e = _np.asarray(expected, dtype=float)
        ok = g.shape == e.shape and bool(_np.allclose(g, e, atol=tol, rtol=tol))
    except Exception:
        ok = False
    return check(name, ok, hint)

def check_hash(name, value, expected_sha, decimals=3, hint=""):
    """Compare an answer against a hidden expected value (only its hash is stored)."""
    try:
        ok = value is not None and sha_of(value, decimals) == expected_sha
    except Exception:
        ok = False
    return check(name, ok, hint)

def summary():
    n_ok = sum(1 for v in _results.values() if v)
    n = len(_results)
    print(f"\nScore: {n_ok}/{n} checks passed" + ("  🎉" if n and n_ok == n else ""))
    if n and n_ok < n:
        print("Not yet passing: " + ", ".join(k for k, v in _results.items() if not v))

## Exercise 1 — The Bernoulli log-likelihood and its maximiser  `★☆☆`

For $k$ successes in $n$ independent Bernoulli trials the log-likelihood (dropping the binomial coefficient,
which does not depend on $p$) is $\ell(p) = k\log p + (n - k)\log(1 - p)$. Implement

* `bernoulli_loglik(p, k, n)` — $\ell(p)$; `p` may be a float or a NumPy array (return an array of the same shape);
* `mle_bernoulli(k, n)` — the maximiser $\hat p$ (derive it by setting $\ell'(p) = 0$; do not search numerically);
* `log_lik_ratio(p1, p2, k, n)` — $\ell(p_1) - \ell(p_2)$ as a float, the log of the likelihood ratio.

Use only NumPy (`np.log`).

In [ ]:
def bernoulli_loglik(p, k, n):
    # YOUR CODE HERE
    return None

def mle_bernoulli(k, n):
    # YOUR CODE HERE
    return None

def log_lik_ratio(p1, p2, k, n):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
try:
    got = bernoulli_loglik(0.5, 7, 10)
except Exception:
    got = None
check_close('log-likelihood at p = 0.5 for 7/10 is 10 log(0.5)', got, 10 * np.log(0.5))
try:
    arr = bernoulli_loglik(np.linspace(0.01, 0.99, 99), 7, 10)
    ok_shape = np.shape(arr) == (99,)
    argmax_p = float(np.linspace(0.01, 0.99, 99)[np.argmax(arr)]) if ok_shape else None
except Exception:
    ok_shape, argmax_p = False, None
check('vectorised over an array of p', ok_shape)
check_close('the grid maximum is at p = 0.70', argmax_p, 0.70, tol=1e-9)
try:
    got = mle_bernoulli(7, 10)
except Exception:
    got = None
check_close('MLE of 7/10 is 0.7', got, 0.7)
try:
    got = log_lik_ratio(0.7, 0.5, 7, 10)
except Exception:
    got = None
check_hash('log likelihood ratio 0.7 vs 0.5 for 7 heads in 10', got, '848375abd3971085')
try:
    got = log_lik_ratio(0.7, 0.5, 70, 100)
except Exception:
    got = None
check_hash('log likelihood ratio 0.7 vs 0.5 for 70 heads in 100', got, '6d1d21a7d2c5af98')

## Exercise 2 — Derive and code the Exponential MLE  `★☆☆`

Waiting times $x_1, \dots, x_n$ follow an Exponential distribution with rate $\lambda$, density
$f(x) = \lambda e^{-\lambda x}$. Derive the log-likelihood, set its derivative to zero, and implement

* `exp_loglik(lam, x)` — $\ell(\lambda)$ for an array `x`, as a float;
* `mle_exponential(x)` — $\hat\lambda$ as a float (closed form, no optimiser);
* `mle_tail_prob(x, t)` — the MLE of $P(X > t) = e^{-\lambda t}$, using the invariance property.

Test data: `x = np.array([2.0, 0.5, 1.5, 4.0])` and `t = 3.0`. The second hidden check draws 400 waiting
times with `rng.exponential(scale=1/2.5, size=400)` right after the setup cell's `rng` is created (so run the
setup cell first, then only this exercise's cells, in order).

In [ ]:
def exp_loglik(lam, x):
    # YOUR CODE HERE
    return None

def mle_exponential(x):
    # YOUR CODE HERE
    return None

def mle_tail_prob(x, t):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
x_rep = np.array([2.0, 0.5, 1.5, 4.0])
try:
    got = mle_exponential(x_rep)
except Exception:
    got = None
check_close('MLE rate for the repair times is 1/mean = 0.5', got, 0.5)
try:
    got = mle_tail_prob(x_rep, 3.0)
except Exception:
    got = None
check_hash('MLE of P(X > 3) by invariance', got, '09184c69b2341ec7', decimals=4)
try:
    ll0 = exp_loglik(0.5, x_rep); ll1 = exp_loglik(0.55, x_rep); ll2 = exp_loglik(0.45, x_rep)
    ok_max = (ll0 is not None) and ll0 > ll1 and ll0 > ll2
except Exception:
    ok_max = False
check('the log-likelihood is larger at the MLE than 10% either side', ok_max)
try:
    got = exp_loglik(0.5, x_rep)
except Exception:
    got = None
check_close('log-likelihood at lambda = 0.5 equals 4 log 0.5 - 4', got, 4 * np.log(0.5) - 4.0)
rng = np.random.default_rng(0)
x_big = rng.exponential(scale=1/2.5, size=400)
try:
    got = mle_exponential(x_big)
except Exception:
    got = None
check_hash('MLE rate for the 400 simulated waiting times', got, '4d8508a3d2fec0b7', decimals=3)
check('that estimate is within 0.3 of the true rate 2.5', (got is not None) and abs(got - 2.5) < 0.3)

## Exercise 3 — Fisher information and a Wald interval for a Poisson rate  `★★☆`

For a single Poisson count $x$ with rate $\lambda$, $\log p(x \mid \lambda) = x\log\lambda - \lambda - \log x!$.
Derive the Fisher information $i(\lambda) = -\mathbb{E}[\partial^2_\lambda \log p]$ by hand, then implement

* `fisher_info_poisson(lam, n)` — $I_n(\lambda) = n\,i(\lambda)$ for $n$ i.i.d. counts, as a float;
* `poisson_mle_se(x)` — a tuple `(lam_hat, se)` with the MLE and its standard error $1/\sqrt{I_n(\hat\lambda)}$;
* `wald_ci_poisson(x, z=1.96)` — the tuple `(lower, upper)` of the Wald interval $\hat\lambda \pm z\,\operatorname{SE}$;
* `observed_info_poisson(x)` — the observed information $J(\hat\lambda) = -\ell''(\hat\lambda)$ computed from the
  data (not from the expectation formula). For the Poisson it should coincide with $I_n(\hat\lambda)$.

Test data: the six hourly call counts `[3, 5, 2, 4, 6, 4]`. A second check uses 250 counts drawn with
`rng.poisson(7.0, size=250)` from a freshly seeded `np.random.default_rng(0)`.

In [ ]:
def fisher_info_poisson(lam, n):
    # YOUR CODE HERE
    return None

def poisson_mle_se(x):
    # YOUR CODE HERE
    return None

def wald_ci_poisson(x, z=1.96):
    # YOUR CODE HERE
    return None

def observed_info_poisson(x):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
calls = np.array([3, 5, 2, 4, 6, 4])
try:
    got = fisher_info_poisson(4.0, 6)
except Exception:
    got = None
check_close('I_6(lambda = 4) = 6/4 = 1.5', got, 1.5)
try:
    lam_hat, se = poisson_mle_se(calls)
except Exception:
    lam_hat, se = None, None
check_close('MLE for the call counts is 4.0', lam_hat, 4.0)
check_hash('standard error of the call-rate MLE', se, '213eb13b19246f1c', decimals=4)
try:
    lo, hi = wald_ci_poisson(calls)
except Exception:
    lo, hi = None, None
check_hash('upper end of the 95% Wald interval for the call rate', hi, 'b21f0796883ff372', decimals=3)
check_close('the interval is symmetric about the MLE', (None if lo is None else (lo + hi) / 2), 4.0)
try:
    got = observed_info_poisson(calls)
except Exception:
    got = None
check_close('observed information equals n/lambda_hat for the call counts', got, 6 / 4.0)
rng = np.random.default_rng(0)
counts_big = rng.poisson(7.0, size=250)
try:
    lo, hi = wald_ci_poisson(counts_big)
except Exception:
    lo, hi = None, None
check_hash('lower end of the 95% Wald interval for the 250 counts', lo, '3b9dd1bbe5d55f05', decimals=3)
check('that interval contains the true rate 7.0', (lo is not None) and lo < 7.0 < hi)

## Exercise 4 — Numerical MLE for the Gamma distribution  `★★☆`

The Gamma$(\alpha, \beta)$ density (shape $\alpha$, **rate** $\beta$) is
$f(x) = \frac{\beta^\alpha}{\Gamma(\alpha)}x^{\alpha - 1}e^{-\beta x}$, so for a sample $x_1, \dots, x_n$

$$\ell(\alpha, \beta) = n\alpha\log\beta - n\log\Gamma(\alpha) + (\alpha - 1)\sum_i\log x_i - \beta\sum_i x_i .$$

There is no closed form for $\hat\alpha$. Implement

* `gamma_nll(theta, x)` — the **negative** log-likelihood as a float, where `theta = (log_alpha, log_beta)` are
  the *logarithms* of the parameters (so the optimiser can roam over all of $\mathbb{R}^2$). Use
  `scipy.special.gammaln` for $\log\Gamma$.
* `fit_gamma_mle(x)` — returns the tuple `(alpha_hat, beta_hat)` by minimising `gamma_nll` with
  `scipy.optimize.minimize` (any method; a method-of-moments start $\alpha_0 = \bar x^2/\hat\sigma^2$,
  $\beta_0 = \alpha_0/\bar x$ is a good idea).

Checks compare with the guide's hand-worked five-point example and with `scipy.stats.gamma.fit(x, floc=0)`
(which returns `shape, loc, scale` with `scale = 1/beta`) on 300 points drawn via
`rng.gamma(shape=2.5, scale=1/1.5, size=300)` from a fresh `np.random.default_rng(0)`.

In [ ]:
def gamma_nll(theta, x):
    log_a, log_b = theta
    # YOUR CODE HERE
    return None

def fit_gamma_mle(x):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
x_small = np.array([1.2, 2.5, 0.8, 3.1, 1.9])
try:
    got = gamma_nll(np.log([2.0, 1.0]), x_small)
except Exception:
    got = None
check_close('NLL at alpha = 2, beta = 1 equals -sum(log x) + sum(x)', got, float(-(np.log(x_small).sum()) + x_small.sum()), tol=1e-6)
try:
    a_s, b_s = fit_gamma_mle(x_small)
except Exception:
    a_s, b_s = None, None
check_close("shape MLE for the guide's five points is 4.620", a_s, 4.6201, tol=2e-3)
check_close('rate MLE equals shape / mean', (None if a_s is None else b_s), (None if a_s is None else a_s / x_small.mean()), tol=2e-3)
rng = np.random.default_rng(0)
x_sim = rng.gamma(shape=2.5, scale=1/1.5, size=300)
try:
    a_hat, b_hat = fit_gamma_mle(x_sim)
    a_sp, _, scale_sp = stats.gamma.fit(x_sim, floc=0)
except Exception:
    a_hat, b_hat, a_sp, scale_sp = None, None, 1.0, 1.0
check_close('shape agrees with scipy.stats.gamma.fit', a_hat, a_sp, tol=1e-3)
check_close('rate agrees with 1/scale from scipy', b_hat, 1 / scale_sp, tol=1e-3)
try:
    nll_mle = gamma_nll(np.log([a_hat, b_hat]), x_sim)
    nll_off = gamma_nll(np.log([a_hat * 1.2, b_hat]), x_sim)
    ok = nll_mle < nll_off
except Exception:
    ok = False
check('the NLL is lower at the fitted parameters than at a perturbed shape', ok)

## Exercise 5 — Cross-entropy is the negative log-likelihood of the labels  `★★☆`

Build the classification losses *from the likelihood*, not from a formula you remember:

* `bce_from_likelihood(y, p)` — for binary labels `y` and predicted probabilities `p`, compute the Bernoulli
  likelihood of each label, $p_i^{y_i}(1 - p_i)^{1 - y_i}$, take logs, and return $-\frac1n\sum_i \log(\cdot)$
  as a float. It must equal the binary cross-entropy.
* `softmax(z)` — row-wise softmax of a 2-D array of logits (subtract the row maximum first for stability).
* `categorical_nll(logits, labels)` — for an $n \times K$ array of logits and integer `labels` in $\{0, \dots, K-1\}$,
  return $-\frac1n\sum_i \log \text{softmax}(z_i)_{y_i}$ as a float.
* `ce_grad_logits(logits, labels)` — the gradient of `categorical_nll` with respect to the logits, an $n \times K$
  array. Derive it: for the mean loss it is $(p_i - \mathbf{e}_{y_i})/n$ in row $i$.

Use only NumPy. The checks compare with `sklearn.metrics.log_loss` and with a finite-difference gradient.

In [ ]:
def bce_from_likelihood(y, p):
    # YOUR CODE HERE
    return None

def softmax(z):
    # YOUR CODE HERE
    return None

def categorical_nll(logits, labels):
    # YOUR CODE HERE
    return None

def ce_grad_logits(logits, labels):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
from sklearn.metrics import log_loss
y_demo = np.array([1, 0, 1, 1, 0]); p_demo = np.array([0.9, 0.2, 0.6, 0.3, 0.1])
try:
    got = bce_from_likelihood(y_demo, p_demo)
except Exception:
    got = None
check_close('BCE from the Bernoulli likelihood equals sklearn log_loss', got, log_loss(y_demo, p_demo), tol=1e-9)
try:
    got = bce_from_likelihood(np.array([1, 1]), np.array([0.5, 0.5]))
except Exception:
    got = None
check_close('two labels at p = 0.5 cost log 2 each', got, np.log(2.0))
logits = np.array([[2.0, 1.0, 0.1], [0.5, 2.5, -1.0], [1.0, 1.0, 1.0], [-1.0, 0.0, 3.0]])
labels = np.array([0, 1, 2, 1])
try:
    sm = softmax(logits)
    ok_rows = np.allclose(sm.sum(axis=1), 1.0) and sm.shape == (4, 3) and np.all(sm > 0)
except Exception:
    ok_rows = False
check('softmax rows are positive and sum to 1', ok_rows)
try:
    got = categorical_nll(logits, labels)
except Exception:
    got = None
check_close('categorical NLL equals sklearn log_loss on the softmax probabilities', got, (log_loss(labels, softmax(logits), labels=[0, 1, 2]) if ok_rows else None), tol=1e-9)
check_hash('categorical NLL of the four examples', got, 'b84896071eba5d87', decimals=4)
try:
    g = ce_grad_logits(logits, labels)
    eps = 1e-6; g_fd = np.zeros_like(logits)
    for i in range(logits.shape[0]):
        for j in range(logits.shape[1]):
            d = np.zeros_like(logits); d[i, j] = eps
            g_fd[i, j] = (categorical_nll(logits + d, labels) - categorical_nll(logits - d, labels)) / (2 * eps)
    ok_grad = g.shape == (4, 3) and np.allclose(g, g_fd, atol=1e-6)
    ok_rowsum = np.allclose(g.sum(axis=1), 0.0, atol=1e-12)
except Exception:
    ok_grad, ok_rowsum = False, False
check('analytic gradient (p - y)/n matches finite differences', ok_grad)
check('each row of the gradient sums to zero', ok_rowsum)
try:
    got = float(ce_grad_logits(logits, labels)[3, 1])
except Exception:
    got = None
check_hash('gradient entry for example 4, true class 1', got, '712660d7fa1c3bdf', decimals=4)

## Exercise 6 — Beta-Binomial posterior, posterior mean and credible interval  `★★☆`

Implement the conjugate update for a success probability:

* `beta_posterior(a, b, k, n)` — the tuple `(a_post, b_post)` of the posterior Beta parameters after $k$
  successes in $n$ trials with a Beta$(a, b)$ prior;
* `beta_posterior_summary(a, b, k, n, level=0.95)` — a dict with keys `a`, `b`, `mean` (posterior mean) and
  `lower`, `upper` (the equal-tailed `level` credible interval, via `scipy.stats.beta.ppf`);
* `prob_rate_above(a, b, k, n, threshold)` — $P(p > \text{threshold} \mid \mathcal{D})$ as a float.

Test cases: (i) uniform prior, 2 clicks in 40 impressions; (ii) Beta$(2, 8)$ prior, 7 heads in 10;
(iii) the surgeon from the guide — Beta$(5, 95)$ prior, 2 complications in 15 operations.

In [ ]:
def beta_posterior(a, b, k, n):
    # YOUR CODE HERE
    return None

def beta_posterior_summary(a, b, k, n, level=0.95):
    # YOUR CODE HERE
    return None

def prob_rate_above(a, b, k, n, threshold):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
try:
    got = beta_posterior(1, 1, 2, 40)
except Exception:
    got = None
check('uniform prior + 2/40 gives Beta(3, 39)', got is not None and tuple(got) == (3, 39))
try:
    s = beta_posterior_summary(1, 1, 2, 40)
    mean, lo, hi = s["mean"], s["lower"], s["upper"]
except Exception:
    mean, lo, hi = None, None, None
check_close('posterior mean 3/42', mean, 3/42)
check_hash('upper end of the 95% credible interval (2 in 40, uniform prior)', hi, '2c98d172582a8634', decimals=4)
check('the interval is inside [0, 1] and contains the posterior mean', (lo is not None) and 0 < lo < mean < hi < 1)
try:
    s = beta_posterior_summary(2, 8, 7, 10)
    got = s["mean"]
except Exception:
    got = None
check_close('Beta(2,8) prior + 7/10 -> mean 0.45', got, 0.45)
try:
    s = beta_posterior_summary(5, 95, 2, 15)
    got = s["mean"]
except Exception:
    got = None
check_hash('shrunken complication rate of the surgeon (Beta(5,95) prior, 2 in 15)', got, '7551fe1e128c1b0a', decimals=4)
try:
    got = prob_rate_above(1, 1, 2, 40, 0.10)
except Exception:
    got = None
check_hash('P(p > 0.10 | 2 clicks in 40, uniform prior)', got, '4eb83c23e3c4dcf9', decimals=3)
try:
    got = prob_rate_above(5, 95, 20, 150, 0.08)
except Exception:
    got = None
check_hash('P(rate > 8%) for the surgeon with 20 complications in 150', got, '2c3b2e9841dcb3e3', decimals=3)

## Exercise 7 — Normal-Normal updating: precisions add  `★★☆`

A quantity $\mu$ has prior $\mathcal{N}(\mu_0, \tau^2)$; observations $x_i \sim \mathcal{N}(\mu, \sigma^2)$ with
**known** $\sigma^2$. Implement

* `normal_normal_posterior(mu0, tau2, sigma2, x)` — returns `(mu_post, var_post)` for the array `x`, using the
  precision form $\frac{1}{\sigma_n^2} = \frac{1}{\tau^2} + \frac{n}{\sigma^2}$,
  $\mu_n = \sigma_n^2\big(\frac{\mu_0}{\tau^2} + \frac{\sum_i x_i}{\sigma^2}\big)$;
* `sequential_normal_update(mu0, tau2, sigma2, x)` — the same posterior obtained by updating **one observation at
  a time**, feeding each posterior in as the next prior; it must return the same `(mu_post, var_post)`.

Test case: prior $\mathcal{N}(100, 10^2)$, sensor noise $\sigma = 4$, readings `[106.0, 110.5, 107.2, 108.3]`.
A second check uses 25 draws `rng.normal(1.5, 2.0, size=25)` from a fresh `np.random.default_rng(0)` with prior
$\mathcal{N}(0, 1)$ and $\sigma^2 = 4$.

In [ ]:
def normal_normal_posterior(mu0, tau2, sigma2, x):
    # YOUR CODE HERE
    return None

def sequential_normal_update(mu0, tau2, sigma2, x):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
readings = np.array([106.0, 110.5, 107.2, 108.3])
try:
    mu_b, var_b = normal_normal_posterior(100.0, 100.0, 16.0, readings)
except Exception:
    mu_b, var_b = None, None
check_close('posterior variance 1/(0.01 + 0.25)', var_b, 1 / 0.26)
check_hash('posterior mean for the furnace readings', mu_b, '9f0a95988c6131dc', decimals=3)
check('posterior mean lies between the prior mean and the sample mean', (mu_b is not None) and 100.0 < mu_b < readings.mean())
try:
    mu_s, var_s = sequential_normal_update(100.0, 100.0, 16.0, readings)
except Exception:
    mu_s, var_s = None, None
check_close('sequential update gives the same mean as the batch update', mu_s, mu_b, tol=1e-9)
check_close('sequential update gives the same variance as the batch update', var_s, var_b, tol=1e-9)
rng = np.random.default_rng(0)
x25 = rng.normal(1.5, 2.0, size=25)
try:
    mu_c, var_c = normal_normal_posterior(0.0, 1.0, 4.0, x25)
except Exception:
    mu_c, var_c = None, None
check_hash('posterior mean for the 25 draws', mu_c, '3b07dd65a821c022', decimals=3)
check_close('posterior variance 1/(1 + 25/4)', var_c, 1 / (1 + 25 / 4))

## Exercise 8 — MAP with a Gaussian prior is ridge regression with lambda = sigma^2 / tau^2  `★★☆`

For $y = Xw + \varepsilon$, $\varepsilon_i \sim \mathcal{N}(0, \sigma^2)$, with prior $w_j \sim \mathcal{N}(0, \tau^2)$,
the guide derives $\hat w_{\text{MAP}} = \arg\min_w \|y - Xw\|^2 + \lambda\|w\|^2$ with $\lambda = \sigma^2/\tau^2$.
Implement, with NumPy only (`np.linalg.solve`, no scikit-learn inside the functions):

* `map_gaussian_prior(X, y, sigma2, tau2)` — the MAP weight vector via the closed form $(X^\top X + \lambda I)^{-1}X^\top y$;
* `mle_least_squares(X, y)` — the MLE / ordinary least-squares weights $(X^\top X)^{-1}X^\top y$.

The checks generate `X8 = rng.normal(size=(40, 6))`, `y8 = X8 @ w_true + rng.normal(0, 0.5, size=40)` with
`w_true = [2, -1, 0.5, 0, 0, 1.5]` from a fresh `np.random.default_rng(0)`, compare your MAP with
`Ridge(alpha=sigma2/tau2, fit_intercept=False)`, and verify two consequences: the MAP has a smaller norm than
the MLE, and stacking the data ten times (same prior) moves the MAP toward the MLE.

In [ ]:
def map_gaussian_prior(X, y, sigma2, tau2):
    # YOUR CODE HERE
    return None

def mle_least_squares(X, y):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
from sklearn.linear_model import Ridge
rng = np.random.default_rng(0)
X8 = rng.normal(size=(40, 6)); w_true = np.array([2.0, -1.0, 0.5, 0.0, 0.0, 1.5])
y8 = X8 @ w_true + rng.normal(0, 0.5, size=40)
try:
    w_map = np.asarray(map_gaussian_prior(X8, y8, sigma2=0.25, tau2=0.01), dtype=float)
    w_mle = np.asarray(mle_least_squares(X8, y8), dtype=float)
    ok_shape = w_map.shape == (6,) and w_mle.shape == (6,)
except Exception:
    w_map, w_mle, ok_shape = None, None, False
check('both weight vectors have shape (6,)', ok_shape)
check_close('MAP equals Ridge(alpha = sigma^2/tau^2 = 25)', w_map, Ridge(alpha=25.0, fit_intercept=False).fit(X8, y8).coef_, tol=1e-8)
check_close('MLE equals np.linalg.lstsq', w_mle, np.linalg.lstsq(X8, y8, rcond=None)[0], tol=1e-8)
check('the MAP has a smaller norm than the MLE (shrinkage)', ok_shape and np.linalg.norm(w_map) < np.linalg.norm(w_mle))
check_hash('norm of the MAP weights', (float(np.linalg.norm(w_map)) if ok_shape else None), '283339cfe5eb32f6', decimals=3)
try:
    w_loose = map_gaussian_prior(X8, y8, sigma2=0.25, tau2=1e6)
    ok_loose = np.allclose(w_loose, w_mle, atol=1e-4)
except Exception:
    ok_loose = False
check('a very wide prior (tau^2 = 1e6) recovers the MLE', ok_loose)
try:
    X80 = np.vstack([X8] * 10); y80 = np.concatenate([y8] * 10)
    w_big = map_gaussian_prior(X80, y80, sigma2=0.25, tau2=0.01)
    ok_big = np.linalg.norm(w_big - w_mle) < np.linalg.norm(w_map - w_mle)
except Exception:
    ok_big = False
check('ten times the data, same prior: the MAP moves toward the MLE', ok_big)

## Exercise 9 — Credible interval or confidence interval?  `★★☆`

A data scientist runs an experiment and reports: "The 95% confidence interval for the conversion rate is
$[4.1\%, 5.9\%]$. So there is a 95% probability that the true rate is between 4.1% and 5.9%." A colleague
replies: "You can only say that if you computed a credible interval — and then you'd have to tell me your
prior." Who is right? In your answer (a) explain precisely what the 95% in a confidence interval refers to,
(b) explain what a 95% credible interval claims and what it requires, (c) say under which circumstances the
two intervals are numerically almost identical, and (d) give one situation from this module where they differ
sharply and say which one you would trust.

<details><summary><b>Show answer</b></summary>

The colleague is right about the logic, even if the numbers may be close. **(a)** A confidence interval is
the output of a *procedure*; "95%" is the long-run frequency with which that procedure captures the fixed,
unknown parameter over hypothetical repetitions of the experiment. Once the data are in and the interval is
$[4.1\%, 5.9\%]$, the parameter is either inside it or not — no probability attaches to this particular
interval. **(b)** A 95% credible interval $[a, b]$ satisfies $P(a \le \theta \le b \mid \mathcal{D}) = 0.95$,
a direct probability statement about the parameter *given these data*; it requires a prior, because without
one there is no posterior to take quantiles of. **(c)** With a flat (or weak) prior and a large sample, the
posterior is approximately $\mathcal{N}(\hat\theta, \operatorname{SE}^2)$ — the same normal that the Wald
interval is built from — so the two intervals nearly coincide numerically (for thousands of visitors, both
are about $\hat p \pm 1.96\sqrt{\hat p(1-\hat p)/n}$). The sentence becomes *correct* if it is read as a
statement about the flat-prior posterior. **(d)** They differ sharply for small $n$ or rates near 0 or 1: for
2 clicks in 40 impressions the Wald interval is $[-1.8\%, 11.8\%]$ — it includes impossible negative rates
and understates the right tail — while the Beta$(3, 39)$ credible interval is $[1.5\%, 16.5\%]$. Here the
credible interval is the one to trust, because it is exact for any $n$ and respects the parameter space,
whereas the Wald interval relies on asymptotics that have not kicked in. (A frequentist would reach for an
exact or Wilson interval instead, [Module 12](../12-hypothesis-testing-and-experiments/README.md).)

</details>

## Exercise 10 — Posterior predictive versus plug-in prediction  `★★☆`

After observing $k$ successes in $n$ trials with a Beta$(a_0, b_0)$ prior, the posterior is Beta$(a, b)$ and the
number of successes $J$ in $m$ *future* trials has the Beta-Binomial predictive distribution

$$P(J = j \mid \mathcal{D}) = \binom{m}{j}\frac{B(j + a,\; m - j + b)}{B(a, b)}, \qquad j = 0, \dots, m,$$

with $B$ the Beta function. Implement, using `scipy.special.betaln` and `gammaln` (work in logs; do **not**
use `scipy.stats.betabinom` — build it yourself):

* `beta_binomial_pmf(j, m, a, b)` — the PMF for an integer or integer array `j`;
* `predictive_interval(a, b, m, level=0.95)` — the tuple `(lo, hi)` of the smallest $j$ with CDF $\ge 0.025$ and the
  smallest $j$ with CDF $\ge 0.975$ (use `np.searchsorted` on the cumulative sum);
* `plug_in_interval(a, b, m, level=0.95)` — the same quantiles of the plug-in Binomial$(m, \bar p)$ with
  $\bar p = a/(a + b)$, via `scipy.stats.binom.ppf`;
* `prob_no_events(a, b, m)` — the predictive probability of zero successes in the next $m$ trials, $\frac{B(a, b + m)}{B(a, b)}$.

Test case: posterior Beta$(3, 39)$, $m = 100$ (the click-through example); and the zero-failure posterior
Beta$(1, 21)$ with $m = 10$.

In [ ]:
def beta_binomial_pmf(j, m, a, b):
    # YOUR CODE HERE
    return None

def predictive_interval(a, b, m, level=0.95):
    # YOUR CODE HERE
    return None

def plug_in_interval(a, b, m, level=0.95):
    # YOUR CODE HERE
    return None

def prob_no_events(a, b, m):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
try:
    pmf = np.asarray(beta_binomial_pmf(np.arange(101), 100, 3, 39), dtype=float)
    ok_pmf = pmf.shape == (101,) and abs(pmf.sum() - 1) < 1e-9 and np.all(pmf >= 0)
except Exception:
    pmf, ok_pmf = None, False
check('the predictive PMF is non-negative and sums to 1', ok_pmf)
check_close('matches scipy.stats.betabinom', pmf, stats.betabinom.pmf(np.arange(101), 100, 3, 39), tol=1e-10)
try:
    got = float(beta_binomial_pmf(7, 100, 3, 39))
except Exception:
    got = None
check_hash('P(J = 7 | Beta(3, 39), m = 100)', got, 'e1079cf5f27832c5', decimals=4)
try:
    lo_pred, hi_pred = predictive_interval(3, 39, 100)
    lo_plug, hi_plug = plug_in_interval(3, 39, 100)
except Exception:
    lo_pred, hi_pred, lo_plug, hi_plug = None, None, None, None
check_hash('upper end of the 95% predictive interval', hi_pred, '4ec9599fc203d176')
check_hash('upper end of the 95% plug-in interval', hi_plug, '3fdba35f04dc8c46')
check('the predictive interval is wider than the plug-in interval', (hi_pred is not None) and (hi_pred - lo_pred) > (hi_plug - lo_plug))
try:
    got = prob_no_events(1, 21, 10)
except Exception:
    got = None
check_close('P(no failure in 10 more tests | Beta(1, 21)) = 21/31', got, 21/31)
try:
    got = prob_no_events(3, 39, 100)
except Exception:
    got = None
check_close('consistency: prob_no_events equals the PMF at j = 0', got, (float(pmf[0]) if ok_pmf else None), tol=1e-10)

## Exercise 11 — Bayesian A/B test by Monte Carlo, with an exact check  `★★★`

Implement

* `bayesian_ab_test(kA, nA, kB, nB, n_draws=200_000, a=1, b=1, rng=None)` — with Beta$(a, b)$ priors on both
  conversion rates, draw `n_draws` samples from each posterior (use `rng.beta`; create
  `rng = np.random.default_rng(0)` when `rng is None`) and return a dict with keys `prob_B_better`
  ($P(p_B > p_A \mid \mathcal{D})$), `expected_loss_B` ($\mathbb{E}[\max(p_A - p_B, 0)]$, what you lose on average by
  shipping B if it is worse), `expected_loss_A` ($\mathbb{E}[\max(p_B - p_A, 0)]$) and `lift_mean` ($\mathbb{E}[p_B - p_A]$).
* `prob_B_better_exact(kA, nA, kB, nB, a=1, b=1)` — the *exact* probability, using the closed form for two
  independent Beta posteriors $p_A \sim \text{Beta}(\alpha_A, \beta_A)$, $p_B \sim \text{Beta}(\alpha_B, \beta_B)$ with
  integer $\alpha_B$:

  $$P(p_B > p_A) = \sum_{j=0}^{\alpha_B - 1}\frac{B(\alpha_A + j,\; \beta_A + \beta_B)}{(\beta_B + j)\,B(1 + j,\; \beta_B)\,B(\alpha_A, \beta_A)} .$$

  Compute each term in log space with `scipy.special.betaln` and exponentiate.

Test cases: the guide's A/B data (A: 120/2400, B: 150/2400) and a small test (A: 5/100, B: 9/100).

In [ ]:
def bayesian_ab_test(kA, nA, kB, nB, n_draws=200_000, a=1, b=1, rng=None):
    rng = np.random.default_rng(0) if rng is None else rng
    # YOUR CODE HERE
    return None

def prob_B_better_exact(kA, nA, kB, nB, a=1, b=1):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
try:
    res = bayesian_ab_test(120, 2400, 150, 2400)
    pBA = res["prob_B_better"]; lossB = res["expected_loss_B"]; lossA = res["expected_loss_A"]
except Exception:
    res, pBA, lossB, lossA = None, None, None, None
check('P(B > A) for the guide data is between 0.96 and 0.98', (pBA is not None) and 0.96 < pBA < 0.98)
check('expected loss of shipping B is below 0.0002', (lossB is not None) and 0 <= lossB < 2e-4)
check('expected loss of keeping A is above 0.011', (lossA is not None) and lossA > 0.011)
check_close('lift_mean equals the difference of posterior means (within MC error)', (None if res is None else res["lift_mean"]), 151/2402 - 121/2402, tol=2e-4)
try:
    got = prob_B_better_exact(120, 2400, 150, 2400)
except Exception:
    got = None
check_hash('exact P(B > A) for the guide data', got, 'dc1dd5d51876dbae', decimals=3)
check('Monte Carlo and exact agree within 0.003', (got is not None) and (pBA is not None) and abs(got - pBA) < 0.003)
try:
    got = prob_B_better_exact(5, 100, 9, 100)
except Exception:
    got = None
check_hash('exact P(B > A) for 5/100 vs 9/100', got, '95556db63ad1c7c9', decimals=3)
try:
    got = prob_B_better_exact(1, 5, 2, 5)
    brute = np.mean(np.random.default_rng(1).beta(2, 5, 400_000) < np.random.default_rng(2).beta(3, 4, 400_000))
except Exception:
    got, brute = None, None
check('exact formula agrees with brute force on a tiny example', (got is not None) and abs(got - brute) < 0.003)
try:
    got = prob_B_better_exact(10, 100, 10, 100)
except Exception:
    got = None
check('symmetric data give P(B > A) = 0.5', (got is not None) and abs(got - 0.5) < 1e-9)

## Exercise 12 — Mini-project: empirical-Bayes shrinkage of 20 store conversion rates  `★★★`

A retailer has 20 stores with very different traffic. Each store's conversion rate is estimated from its own
visits, and head office wants to rank the stores and spot genuinely strong or weak ones. Raw rates $k_j/n_j$
are noisy for small stores; empirical Bayes borrows strength across stores by estimating a Beta prior *from
the ensemble* and shrinking each store toward it.

**The model** (use `rng = np.random.default_rng(11)`):

1. Draw 20 true rates `true_rates = rng.beta(6, 114, size=20)` (mean 5%), visit counts
   `visits = rng.integers(20, 400, size=20)`, and `conversions = rng.binomial(visits, true_rates)`.
2. Fit a Beta$(a, b)$ prior by the **method of moments**: let $m$ be the mean of the raw rates and let $v$ be their
   variance (`ddof=1`) **minus** the average binomial sampling variance $\overline{\hat p_j(1 - \hat p_j)/n_j}$
   (floor $v$ at $10^{-6}$); then $a + b = \frac{m(1 - m)}{v} - 1$, $a = m(a + b)$, $b = (1 - m)(a + b)$.
3. Shrink each store to its posterior mean $\frac{a + k_j}{a + b + n_j}$.

**Deliverables**

* the fitted prior and its strength $a + b$ in pseudo-visits, with a one-sentence interpretation;
* the mean squared error of the raw and of the shrunken estimates **against the true rates**, and the MSE of
  the fully pooled estimate (every store $= m$) for comparison;
* a table or plot showing how far each store moves, with the data weight $n_j/(a + b + n_j)$, demonstrating
  that small stores move a lot and large stores barely;
* which store is "best" by raw rate, by shrunken rate, and in truth — and a comment on why they can differ;
* a robustness check: repeat the whole experiment for at least 100 random seeds and report the median ratio of
  shrunken-to-raw MSE and the fraction of replications in which shrinkage helps.

**Rubric — you have mastered this when you can:**

* The sampling-noise correction is applied before fitting the prior (subtracting the average $\hat p(1-\hat p)/n$),
  and the prior strength $a + b$ is reported in pseudo-visits.
* Shrinkage is the Beta posterior mean $\frac{a + k_j}{a + b + n_j}$ — a weighted average with data weight
  $n_j/(a + b + n_j)$ — and the per-store movement is shown to depend on $n_j$.
* MSE is computed against the *true* rates (this is synthetic data; that is the point) for raw, shrunken and
  fully pooled estimates, and the ordering is explained: pooling throws away real differences, raw estimates
  keep all the noise, shrinkage sits between.
* The ranking comparison identifies that the raw "best" store is typically a small store with a lucky sample.
* The robustness loop reports a median MSE ratio and a success fraction over $\ge 100$ seeds, with reproducible
  seeding.
* Figures have titles and axis labels.

In [ ]:
# Your mini-project workspace

## Score

In [ ]:
summary()